# Laboratorium: Kodowanie

### Zadanie 1:Dowód nierówności Krafta i Kodowanie Shannona (2 punkty)

**Wstęp teoretyczny:** Zgodnie z wykładem, kodowanie Shannona polega na przypisaniu symbolom długości $l_i = \lceil -\log_2(p_i) \rceil$. Nierówność Krafta gwarantuje, że dla takich długości można skonstruować jednoznacznie dekodowalny kod prefiksowy.

**Polecenie:**
1. Napisz funkcję w Pythonie, która na wejściu przyjmuje słownik prawdopodobieństw wystąpień symboli (np. `{'A': 0.4, 'B': 0.3, 'C': 0.2, 'D': 0.1}`), a następnie:
   * Wylicza oczekiwane długości słów kodowych $l_i$ dla kodowania Shannona.
   * Sprawdza, czy dla wyliczonych długości zachodzi nierówność Krafta (zwraca wartość logiczną oraz sumę $\sum 2^{-l_i}$).
   * Implementuje algorytm oparty na konstruktywnym dowodzie z wykładu, aby wygenerować konkretne słowa binarne o zadanych długościach $l_i$, tworząc poprawny kod prefiksowy.
2. Zwróć wygenerowany słownik kodów i porównaj empiryczną średnią długość słowa kodowego ($L = \sum p_i l_i$) z teoretyczną entropią Shannona $H(X)$. Zweryfikuj, czy zachodzi nierówność $H(X) \leq L < H(X) + 1$.

In [9]:
import numpy as np


def shannon_coding(probabilities: dict[str, float]) -> dict:
    symbols = np.array(list(probabilities.keys()))
    probs = np.array(list(probabilities.values()), dtype=np.float64)
    
    # Zadanie 1
    lengths_array = np.ceil(-np.log2(probs)).astype(int)
    lengths = dict(zip(symbols, lengths_array))
    
    # Zadanie 2
    kraft_sum = float(np.sum(2.0 ** (-lengths_array)))
    kraft_holds = kraft_sum <= 1
    
    # Zadanie 3
    sorted_indices = np.argsort(lengths_array)
    sorted_symbols = symbols[sorted_indices]
    sorted_lengths = lengths_array[sorted_indices]
    
    codes = {}
    code = 0
    prev_length = 0
    for symbol, length in zip(sorted_symbols, sorted_lengths):
        code <<= (length - prev_length)
        codes[symbol] = format(code, f"0{length}b")
        code += 1
        prev_length = length
    
    # Zadanie 4
    entropy = float(-np.sum(probs * np.log2(probs)))
    avg_length = float(np.sum(probs * lengths_array))
    
    return {
        "lengths": lengths,
        "kraft_sum": kraft_sum,
        "kraft_holds": kraft_holds,
        "codes": codes,
        "entropy": entropy,
        "avg_length": avg_length,
    }


def is_prefix_code(codes: dict[str, str]) -> bool:
    values = list(codes.values())
    for i, a in enumerate(values):
        for j, b in enumerate(values):
            if i != j and b.startswith(a):
                return False
    return True

# Main
probs = {"A": 0.4, "B": 0.3, "C": 0.2, "D": 0.1}
result = shannon_coding(probs)

print("=== Kodowanie Shannona ===")
print(f"Rozkład P: {probs}\n")

print("Długości słów kodowych  l_i = ceil(-log2(p_i)):")
for s, l in result["lengths"].items():
    p_val = probs[s]
    print(f"  l({s}) = {l}")

print(f"\nCzy spełniona nierówność Krafta (<= 1)? {result['kraft_holds']}")

print("\nWygenerowany kod prefiksowy:")
for s, c in result["codes"].items():
    print(f"  {s} -> {c}")

print(f"\nKod jest prefiksowy (weryfikacja): {is_prefix_code(result['codes'])}")

H = result["entropy"]
L = result["avg_length"]
print(f"\nEntropia Shannona  H(X) = {H:.4f} bit/symbol")
print(f"Średnia długość    L    = {L:.4f} bit/symbol")
print(f"Sprawdzenie  H(X) <= L < H(X) + 1 :  {H <= L < H + 1}")
print(f"   {H:.4f} ≤ {L:.4f} < {H + 1:.4f}")


=== Kodowanie Shannona ===
Rozkład P: {'A': 0.4, 'B': 0.3, 'C': 0.2, 'D': 0.1}

Długości słów kodowych  l_i = ceil(-log2(p_i)):
  l(A) = 2
  l(B) = 2
  l(C) = 3
  l(D) = 4

Czy spełniona nierówność Krafta (<= 1)? True

Wygenerowany kod prefiksowy:
  A -> 00
  B -> 01
  C -> 100
  D -> 1010

Kod jest prefiksowy (weryfikacja): True

Entropia Shannona  H(X) = 1.8464 bit/symbol
Średnia długość    L    = 2.4000 bit/symbol
Sprawdzenie  H(X) <= L < H(X) + 1 :  True
   1.8464 ≤ 2.4000 < 2.8464


### Zadanie 2: Optymalność Kodowania Huffmana (3 punkty)

**Wstęp teoretyczny:** Kodowanie Huffmana jest optymalnym kodem prefiksowym (osiąga najmniejszą możliwą średnią długość kodu dla danego rozkładu, będącą najbliżej granicy $H(X)$).

**Polecenie:**
1. Zaimplementuj od zera algorytm Huffmana. Funkcja powinna przyjmować tekst, a zwracać:
   * Słownik kodowy (mapowanie znak $\rightarrow$ kod binarny).
   * Zakodowany ciąg bitów.
   * *Wymóg:* Do budowy drzewa użyj kolejki priorytetowej (np. moduł `heapq`), aby zagwarantować teoretyczną złożoność $O(n \log n)$.
2. Zaimplementuj funkcję dekodującą, która przyjmuje zakodowany ciąg bitów oraz korzeń drzewa Huffmana i odzyskuje oryginalny tekst, przechodząc po węzłach od korzenia do liści. Nie używaj prostego wyszukiwania w słowniku – dekodowanie musi naśladować przejście binarne po drzewie.
3. Wygeneruj losowy tekst o długości 100 000 znaków z alfabetu $\{A, B, C, D, E, F\}$ o zadanym, silnie skośnym rozkładzie (np. $P(A)=0.5, P(B)=0.25, \dots$). Zakoduj go dwoma metodami: swoim kodowaniem Shannona z Zadania 1 oraz kodowaniem Huffmana z Zadania 2. Udowodnij numerycznie własność z wykładu: $L_{Huffman} \leq L_{Shannon}$.

In [10]:
import heapq
from collections import Counter
import random
from typing import Optional

# Definicja węzła drzewa Huffmana
class Node:
    def __init__(self, char: Optional[str] = None, freq: int = 0, left: Optional['Node'] = None, right: Optional['Node'] = None):
        self.char = char
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other: 'Node') -> bool:
        return self.freq < other.freq

def generate_codes(node: Optional[Node], prefix: str = "") -> dict[str, str]:
    if node is None:
        return {}

    if node.char is not None:
        return {node.char: prefix or "0"}

    codes = {}
    codes.update(generate_codes(node.left, prefix + "0"))
    codes.update(generate_codes(node.right, prefix + "1"))
    return codes

# Zadanie 1
def build_huffman_tree(text: str) -> Node:
    frequencies = Counter(text)
    heap = [Node(char=char, freq=freq) for char, freq in frequencies.items()]
    heapq.heapify(heap)

    if len(heap) == 1:
        return heap[0]

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        parent = Node(freq=left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, parent)

    return heap[0]

def huffman_coding(text: str) -> tuple[dict[str, str], str]:
    if not text:
        return {}, ""

    tree = build_huffman_tree(text)
    codes = generate_codes(tree)
    encoded = "".join(codes[char] for char in text)
    return codes, encoded

# Zadanie 2
def huffman_decode(encoded_bits: str, root: Node) -> str:
    if not encoded_bits:
        return ""

    decoded_chars = []
    current = root

    for bit in encoded_bits:
        if bit == "0":
            current = current.left
        else:
            current = current.right

        if current.char is not None:
            decoded_chars.append(current.char)
            current = root

    return "".join(decoded_chars)

# Main
sample_text = "ABRACADABRA"
tree = build_huffman_tree(sample_text)
codes = generate_codes(tree)
encoded = "".join(codes[char] for char in sample_text)
decoded = huffman_decode(encoded, tree)

print("Tekst:")
print(sample_text)
print("\nKody:")
for char in sorted(codes):
    print(f"{char}: {codes[char]}")
print("\nZakodowany tekst:")
print(encoded)
print("\nOdkodowany tekst:")
print(decoded)

# Zadanie 3
print("\nPorównanie Shannon vs Huffman")

random.seed(2137)
probabilities_large = {
    'A': 0.5,
    'B': 0.25,
    'C': 0.1,
    'D': 0.075,
    'E': 0.05,
    'F': 0.025,
}
alphabet = list(probabilities_large.keys())
weights = list(probabilities_large.values())
large_text = ''.join(random.choices(alphabet, weights=weights, k=100000))

shannon_result = shannon_coding(probabilities_large)
shannon_codes = shannon_result['codes']
shannon_encoded = ''.join(shannon_codes[ch] for ch in large_text)

huffman_tree_large = build_huffman_tree(large_text)
huffman_codes_large = generate_codes(huffman_tree_large)
huffman_encoded_large = ''.join(huffman_codes_large[ch] for ch in large_text)

print(f"Długość kodu Shannona: {len(shannon_encoded)}")
print(f"Długość kodu Huffmana: {len(huffman_encoded_large)}")
print(f"Huffman <= Shannon? {len(huffman_encoded_large) <= len(shannon_encoded)}")


Tekst:
ABRACADABRA

Kody:
A: 0
B: 111
C: 101
D: 100
R: 110

Zakodowany tekst:
01111100101010001111100

Odkodowany tekst:
ABRACADABRA

Porównanie Shannon vs Huffman
Długość kodu Shannona: 210113
Długość kodu Huffmana: 197651
Huffman <= Shannon? True


### Zadanie 3: Asymmetric Numeral Systems (ANS) w praktyce (4 punkty)

**Wstęp teoretyczny:** ANS to metoda kompresji, w której cały stan zakodowanej wiadomości trzymany jest w jednej liczbie naturalnej $x$.

**Polecenie:**
Bazując na matematycznym przykładzie z wykładu dla alfabetu $A = \{A, B\}$ z częstościami $f(A) = 3$, $f(B) = 1$ oraz $M = 4$:
1. Zaimplementuj klasę `ANSCoder`, posiadającą wewnętrzny stan `x` (inicjowany wartością 1).
2. Zaimplementuj metodę `encode_symbol(self, symbol)`, która transformuje stan $x \rightarrow x'$ według wzorów:
   * Dla `A`: $x' = \lfloor \frac{x}{3} \rfloor \cdot 4 + (x \bmod 3)$
   * Dla `B`: $x' = x \cdot 4 + 3$
3. Zaimplementuj metodę `decode_symbol(self)`, która na podstawie wartości $(x \bmod 4)$ rozpoznaje symbol, a następnie odtwarza poprzedni stan $x$, wykonując operacje odwrotne.
4. Przetestuj koder na ciągu znaków (np. `A B A A B A A A B`) podając go na wejście symbol po symbolu, wypisując rosnącą liczbę $x$ po każdym kroku, a następnie zdekoduj ten stan do tyłu.
5. **Analiza:** Policz ile bitów w pamięci zajmuje końcowa wartość $x$ (np. za pomocą metody `x.bit_length()`). Podziel tę wartość przez liczbę zakodowanych symboli i skomentuj wynik, porównując go z entropią rozkładu $(0.75, 0.25)$.

In [11]:
class ANSCoder:
    def __init__(self) -> None:
        self.x = 1

    def encode_symbol(self, symbol: str) -> None:
        if symbol == "A":
            self.x = (self.x // 3) * 4 + (self.x % 3)
        elif symbol == "B":
            self.x = self.x * 4 + 3

    def decode_symbol(self) -> str:
        M = 4
        residue = self.x % M
        if residue == 3:
            q = self.x // M
            prev_x = q * 1 + (residue - 3)
            self.x = prev_x
            return 'B'
        else:
            q = self.x // M
            prev_x = q * 3 + residue
            self.x = prev_x
            return 'A'

coder = ANSCoder()
sequence = "ABAABAAAB"
print(f"start {coder.x}")
for symbol in sequence:
    coder.encode_symbol(symbol)
    print(f"after {symbol}: {coder.x}")

final_x = coder.x
decoded = []
while len(decoded) < len(sequence):
    decoded.append(coder.decode_symbol())
    print(f"decoded {decoded[-1]} state {coder.x}")

decoded_str = ''.join(reversed(decoded))
print(f"\nwejście:    {sequence}")
print(f"zdekodowane: {decoded_str}")

H_ans = -0.75 * np.log2(0.75) - 0.25 * np.log2(0.25)
bits_per_symbol = final_x.bit_length() / len(sequence)
print(f"\nkońcowe x: {final_x}")
print(f"bity: {final_x.bit_length()}")
print(f"bity/symbol: {bits_per_symbol:.4f}")
print(f"entropia H(P): {H_ans:.4f}")
print(f"\nKomentarz: Dla 9 symboli otrzymujemy 1 bit/symbol zamiast 0.8113.")
print(f"Entropia 0.8113 to asymptotyczne ograniczenie osiągalne dla długich ciągów.")
print(f"Na małej próbce dyskretne kodowanie nie może osiągnąć pełnej kompresji teoretycznej.")


start 1
after A: 1
after B: 7
after A: 9
after A: 12
after B: 51
after A: 68
after A: 90
after A: 120
after B: 483
decoded B state 120
decoded A state 90
decoded A state 68
decoded A state 51
decoded B state 12
decoded A state 9
decoded A state 7
decoded B state 1
decoded A state 1

wejście:    ABAABAAAB
zdekodowane: ABAABAAAB

końcowe x: 483
bity: 9
bity/symbol: 1.0000
entropia H(P): 0.8113

Komentarz: Dla 9 symboli otrzymujemy 1 bit/symbol zamiast 0.8113.
Entropia 0.8113 to asymptotyczne ograniczenie osiągalne dla długich ciągów.
Na małej próbce dyskretne kodowanie nie może osiągnąć pełnej kompresji teoretycznej.


### Zadanie 4: Kara za niewiedzę – Dywergencja KL i Entropia Krzyżowa (1 punkt)

**Wstęp teoretyczny:** Użycie kodu optymalnego dla rozkładu $Q$, podczas gdy dane w rzeczywistości pochodzą z rozkładu $P$, skutkuje nieoptymalną kompresją. Różnica ta wynosi dokładnie $D_{KL}(P \parallel Q)$.

**Polecenie:**
1. Zdefiniuj dwa rozkłady prawdopodobieństwa dla alfabetu $\{a, b, c, d\}$:
   * Rzeczywisty rozkład źródła $P$: `{'a': 0.5, 'b': 0.25, 'c': 0.125, 'd': 0.125}`
   * Zakładany (błędny) model $Q$: `{'a': 0.125, 'b': 0.125, 'c': 0.25, 'd': 0.5}`
2. Oblicz z poziomu kodu teoretyczne wartości: $H(P)$, Entropię krzyżową $H(P, Q)$ oraz Dywergencję Kullbacka-Leiblera $D_{KL}(P \parallel Q)$ korzystając ze wzorów z wykładu. Upewnij się (np. używając `assert`), czy zachodzi $H(P, Q) = H(P) + D_{KL}(P \parallel Q)$.

In [14]:
import numpy as np

P = np.array([0.5, 0.25, 0.125, 0.125])
Q = np.array([0.125, 0.125, 0.25, 0.5])

H_P = -np.sum(P * np.log2(P))
H_PQ = -np.sum(P * np.log2(Q))
D_KL = np.sum(P * np.log2(P / Q))

print(f"H(P) = {H_P:.4f}")
print(f"H(P,Q) = {H_PQ:.4f}")
print(f"D_KL(P||Q) = {D_KL:.4f}")
print(f"H(P) + D_KL = {H_P + D_KL:.4f}")

assert abs(H_PQ - (H_P + D_KL)) < 1e-10, "Relacja H(P,Q) = H(P) + D_KL nie zachodzi!"
print("✓ H(P,Q) = H(P) + D_KL(P||Q) potwierdzona")


H(P) = 1.7500
H(P,Q) = 2.6250
D_KL(P||Q) = 0.8750
H(P) + D_KL = 2.6250
✓ H(P,Q) = H(P) + D_KL(P||Q) potwierdzona
